<a href="https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Write the rule in plain words first. Then the reason codes it can output.

Rule

Our baseline rule identifies pages that are likely to gain additional organic clicks by improving their search result appearance rather than creating new content.

The rule prioritizes pages that:

Receive a high number of Google Search impressions. Rank within positions where CTR improvements are realistic (approximately positions 3–15). Have a lower CTR than expected for their ranking position.

These pages already receive visibility but are not attracting enough clicks, making them good candidates for title and meta description optimization.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Setup
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Set HF_TOKEN in Colab Secrets or the environment before running this notebook."
    )

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"

MID_PANEL_PATH = (
    f"{DATASET_URL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

FEATURES = [
    "clicks_7d_avg",
    "impressions_7d_avg",
    "position_7d_avg",
    "ctr_7d",
    "is_weekend",
]

TARGET = "target_next_day_clicks"

print("Connected to the March 2026 mid-panel.")

Connected to the March 2026 mid-panel.


In [ ]:
query = f'''
WITH daily_data AS (

    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,

        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available,

        LEAD(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS target_next_day_clicks,

        LEAD(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS trap_next_day_impressions

    FROM read_parquet('{MID_PANEL_PATH}')

    WHERE gsc_data_available IS TRUE
),

feature_rows AS (

    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        AVG(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS clicks_7d_avg,

        AVG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS impressions_7d_avg,

        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS position_7d_avg,

        (
            SUM(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) * 1.0
            /
            NULLIF(
                SUM(gsc_impressions) OVER (
                    PARTITION BY client_hash_id, content_hash_id
                    ORDER BY report_date
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ),
                0
            )
        ) AS ctr_7d,

        CASE
            WHEN DAYOFWEEK(report_date) IN (0, 6)
            THEN 1
            ELSE 0
        END AS is_weekend,

        target_next_day_clicks,
        trap_next_day_impressions

    FROM daily_data
)

SELECT *
FROM feature_rows

WHERE target_next_day_clicks IS NOT NULL

ORDER BY
    report_date,
    client_hash_id,
    content_hash_id

LIMIT 500000
'''

df = con.sql(query).df()

df["clicks_7d_avg"] = df["clicks_7d_avg"].fillna(0.0)

df["impressions_7d_avg"] = (
    df["impressions_7d_avg"]
    .fillna(0.0)
)

df["position_7d_avg"] = (
    df["position_7d_avg"]
    .fillna(100.0)
)

df["ctr_7d"] = (
    df["ctr_7d"]
    .fillna(0.0)
    .clip(lower=0)
)

df["is_weekend"] = df["is_weekend"].astype(int)

print("Feature vector shape:", df.shape)

display(
    df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id"
        ]
        + FEATURES
        + [TARGET]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (500000, 10)


,report_date,client_hash_id,content_hash_id,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,is_weekend,target_next_day_clicks
0,2026-03-01,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.0,6.0,6.166667,0.0,1,0
1,2026-03-01,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,0.0,7.0,8.714286,0.0,1,0
2,2026-03-01,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,0.0,4.0,8.750000,0.0,1,0
3,2026-03-01,client_0797ff3a1fc9a6a5,content_1207efddce873942,0.0,15.0,20.866667,0.0,1,0
4,2026-03-01,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,0.0,3.0,10.666667,0.0,1,0


In [ ]:
baseline = df.copy()

# Simple historical baseline
baseline["baseline_pred"] = (
    baseline["clicks_7d_avg"]
)

In [ ]:
baseline["ctr_gap_score"] = (
    1 / (baseline["ctr_7d"] + 0.01)
)

baseline["visibility_score"] = (
    np.log1p(
        baseline["impressions_7d_avg"]
    )
)

baseline["position_score"] = (
    np.log1p(
        baseline["position_7d_avg"].clip(lower=1)
    )
)

baseline["action_score"] = (
    baseline["visibility_score"]
    * baseline["ctr_gap_score"]
    * (1 + 0.1 * baseline["position_score"])
    * np.log1p(baseline["clicks_7d_avg"])
)

In [ ]:
baseline["reason_code"] = np.select(
    [
        (
            baseline["impressions_7d_avg"]
            > baseline["impressions_7d_avg"].quantile(0.75)
        )
        &
        (
            baseline["ctr_7d"]
            < baseline["ctr_7d"].quantile(0.25)
        ),

        baseline["position_7d_avg"]
        > baseline["position_7d_avg"].quantile(0.75),

        baseline["clicks_7d_avg"]
        < baseline["clicks_7d_avg"].quantile(0.25),
    ],

    [
        "HIGH_VISIBILITY_LOW_CTR",
        "WEAKER_POSITION",
        "LOW_RECENT_CLICKS",
    ],

    default="GENERAL_OPPORTUNITY"
)

In [ ]:
baseline["action"] = np.select(
    [
        baseline["reason_code"]
        == "HIGH_VISIBILITY_LOW_CTR",

        baseline["reason_code"]
        == "WEAKER_POSITION",

        baseline["reason_code"]
        == "LOW_RECENT_CLICKS",
    ],

    [
        "Review title/meta CTR opportunity",
        "Review ranking/content relevance",
        "Monitor before major intervention",
    ],

    default="Prioritize for review"
)

display(
    baseline[
        [
            "baseline_pred",
            "action_score",
            "reason_code",
            "action"
        ]
    ].head()
)

,baseline_pred,action_score,reason_code,action
0,0.0,0.0,GENERAL_OPPORTUNITY,Prioritize for review
1,0.0,0.0,GENERAL_OPPORTUNITY,Prioritize for review
2,0.0,0.0,GENERAL_OPPORTUNITY,Prioritize for review
3,0.0,0.0,WEAKER_POSITION,Review ranking/content relevance
4,0.0,0.0,GENERAL_OPPORTUNITY,Prioritize for review


In [ ]:
dates = (
    pd.to_datetime(
        baseline["report_date"]
    )
    .sort_values()
    .unique()
)

cutoff = dates[
    int(len(dates) * 0.8)
]

train = baseline[
    pd.to_datetime(
        baseline["report_date"]
    ) < cutoff
]

test = baseline[
    pd.to_datetime(
        baseline["report_date"]
    ) >= cutoff
]

y_test = test[TARGET]

pred_test = test["baseline_pred"]

mae = mean_absolute_error(
    y_test,
    pred_test
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_test
    )
)

r2 = r2_score(
    y_test,
    pred_test
)

print(
    f"Time cutoff: "
    f"{pd.Timestamp(cutoff).date()}"
)

print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")

print(f"Baseline MAE:  {mae:.4f}")
print(f"Baseline RMSE: {rmse:.4f}")
print(f"Baseline R²:   {r2:.4f}")

Time cutoff: 2026-03-05
Train rows: 419,905
Test rows:  80,095
Baseline MAE:  0.2235
Baseline RMSE: 0.6710
Baseline R²:   0.6129


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = (
    test
    .sort_values(
        "action_score",
        ascending=False
    )
    .loc[
        :,
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "clicks_7d_avg",
            "impressions_7d_avg",
            "position_7d_avg",
            "ctr_7d",
            "baseline_pred",
            "action_score",
            "reason_code",
            "action"
        ]
    ]
    .head(20)
    .reset_index(drop=True)
)

display(top20)

print("\nReason-code mix:")

display(
    top20["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .to_frame("count")
)

,report_date,client_hash_id,content_hash_id,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,baseline_pred,action_score,reason_code,action
0,2026-03-05,client_23a62021009f63c4,content_e8a52cf3d5988c07,29.0,9637.8,15.485458,0.003009,29.0,3070.576991,GENERAL_OPPORTUNITY,Prioritize for review
1,2026-03-05,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,73.4,8947.0,4.385326,0.008204,73.4,2516.754610,GENERAL_OPPORTUNITY,Prioritize for review
2,2026-03-05,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,17.4,6769.8,3.693508,0.002570,17.4,2359.530049,GENERAL_OPPORTUNITY,Prioritize for review
3,2026-03-05,client_73cda7b4e4f265ea,content_471d9cabce329a66,23.6,6008.2,3.435056,0.003928,23.6,2298.841593,GENERAL_OPPORTUNITY,Prioritize for review
4,2026-03-05,client_62f4a7e64f5e0096,content_7172a7fad43f0998,26.4,5770.6,3.137620,0.004575,26.4,2246.555466,GENERAL_OPPORTUNITY,Prioritize for review
5,2026-03-05,client_23a62021009f63c4,content_3df3f32f3fd58dea,9.2,5100.0,26.044516,0.001804,9.2,2233.535905,WEAKER_POSITION,Review ranking/content relevance
6,2026-03-05,client_62f4a7e64f5e0096,content_f107e54b10b43725,44.4,6074.2,2.772021,0.007310,44.4,2175.306183,GENERAL_OPPORTUNITY,Prioritize for review
7,2026-03-05,client_23a62021009f63c4,content_36e53e9c707674fc,5.8,6946.2,33.607433,0.000835,5.8,2119.711710,WEAKER_POSITION,Review ranking/content relevance
8,2026-03-05,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,19.4,4901.8,3.272570,0.003958,19.4,2102.487438,GENERAL_OPPORTUNITY,Prioritize for review
9,2026-03-05,client_62f4a7e64f5e0096,content_922f95c9418ff0ec,17.8,4180.0,4.895119,0.004258,17.8,2020.111745,GENERAL_OPPORTUNITY,Prioritize for review



Reason-code mix:


,count
reason_code,
GENERAL_OPPORTUNITY,15
WEAKER_POSITION,5


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

Weak Picks

The baseline rule is less reliable for:

Pages with very low impressions because CTR estimates are unstable. Newly published pages that have insufficient historical search data. Seasonal or event-driven pages where temporary traffic patterns may distort the score. Brand-specific queries whose CTR behavior differs from general search queries. Leakage Check

The baseline rule does not use:

Future click or impression data. Labels derived from future outcomes. Existing FlyRank product flags. Any information unavailable at the decision point.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.